In [ ]:
%env ALABOS_CONFIG_PATH=system/alabos_config_example.toml
%env SIM_MODE_FLAG=True

In [ ]:
from alab_management.builders import ExperimentBuilder

from alab_gpss.system.tasks.add_sample import GPSSAddSample
from alab_gpss.system.tasks.heating import GPSSHeating
from alab_gpss.system.tasks.powder_dispensing import GPSSPowderDispensing
from alab_gpss.system.tasks.powder_mixing import GPSSPowderMixing
from alab_gpss.system.tasks.remove_sample import RemoveSample
from alab_gpss.system.tasks.sample_grinding_xrd import GPSSSampleGrindingXRD

from alab_gpss.experiment_design.reactions.balance import generate_recipe

In [ ]:
def to_tuple(obj):
    """Convert a nested list to a tuple"""
    if isinstance(obj, list):
        return tuple(to_tuple(item) for item in obj)
    return obj

In [ ]:
import time
today = time.strftime("%m%d%y")

exp = ExperimentBuilder(name=f"halide_project_{today}", tags=["benchmark_run"])

targets_dict = {
    "Li3YCl6": {
        "heating_profile": [[450, 2, 60 * 12]],
        "precursors": ["LiCl", "YCl3"],
    },
}

# group by heating profile
heating_profile_groups = {}
for target, target_info in targets_dict.items():
    heating_profile = to_tuple(target_info["heating_profile"])
    if heating_profile not in heating_profile_groups:
        heating_profile_groups[heating_profile] = []
    heating_profile_groups[heating_profile].append(target)

for heating_profile, targets in heating_profile_groups.items():
    samples = []
    for target in targets:
        target_info = targets_dict[target]
        sample = exp.add_sample(name=f"{target.replace('.', 'p')}_{today}", tags=["benchmark_run"])
        recipe = generate_recipe(target, target_info["precursors"], target_mass_g=0.5)
        samples.append(sample)
        print(recipe)
        add_sample = GPSSAddSample(notify_user=False)
        add_sample.add_to(sample)
        powder_dispensing = GPSSPowderDispensing({p.name: p.mass for p in recipe.precursors}, 1, num_balls=4)
        powder_dispensing.add_to(sample)
        powder_mixing = GPSSPowderMixing([1000, 1500], [300, 300], interval_seconds=30)
        powder_mixing.add_to(sample)

    heating = GPSSHeating(heating_profile)
    heating.add_to(samples)

    for sample in samples:
        sample_grinding_xrd = GPSSSampleGrindingXRD(360, 28)
        sample_grinding_xrd.add_to(sample)
        remove_sample = RemoveSample()
        remove_sample.add_to(sample)

In [ ]:
exp.plot()

In [ ]:
exp.to_dict()

In [ ]:
exp.submit(address="http://localhost:8895")